In [3]:
# AUTOMATIZOVANÉ STAŽENÍ A VYKRESLENÍ PARCEL (RÚIAN / WFS INSPIRE)

import re
import requests
import pandas as pd
from lxml import etree
from pyproj import Transformer
import folium
from shapely.geometry import Polygon
from branca.element import MacroElement
from jinja2 import Template
from IPython.display import display
from folium.map import Layer
from jinja2 import Template


# =============================================================================
# 1. TŘÍDY A FUNKCE PRO VYKRESLOVÁNÍ MAPY (FOLIUM)
# =============================================================================

class BindClickRemove(MacroElement):
    """
    MacroElement, který navěsí click handler na GeoJson vrstvu.
    Po kliknutí na polygon odstraní samotný polygon i k němu příslušející
    textový popisek (marker) z jejich příslušných FeatureGroup vrstev.
    """
    def __init__(self, fg_poly_name: str, fg_text_name: str, gj_name: str, mk_name: str):
        super().__init__()
        self.fg_poly_name = fg_poly_name
        self.fg_text_name = fg_text_name
        self.gj_name = gj_name
        self.mk_name = mk_name

        self._template = Template(
            """
            {% macro script(this, kwargs) %}
            // Navěšení události click -> odstranění polygonu i popisku
            {{ this.gj_name }}.on('click', function(e) {
                {{ this.fg_poly_name }}.removeLayer({{ this.gj_name }});
                {{ this.fg_text_name }}.removeLayer({{ this.mk_name }});
            });
            {% endmacro %}
            """
        )

# Paleta výrazných barev pro vizuální odlišení parcel podle čísla LV
DISTINCT_COLORS = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#46f0f0", "#f032e6", "#bcf60c", "#fabebe",
    "#008080", "#e6beff", "#9a6324", "#fffac8", "#800000",
    "#aaffc3", "#808000", "#ffd8b1", "#000075", "#808080"
]

def plot_parcels_on_map(df_parcel_data: pd.DataFrame) -> folium.Map:
    """
    Vykreslí parcely z DataFrame do interaktivní mapy Folium.
    Obsahuje oddělené vrstvy pro polygony a text (umožňuje vypínání textu).
    """
    transformer = Transformer.from_crs("EPSG:5514", "EPSG:4326", always_xy=True)

    # Ochrana: pokud chybí sloupec LV, doplníme zástupnou hodnotu
    if "LV" not in df_parcel_data.columns:
        df_parcel_data["LV"] = "Neznámé"

    # Zmapování unikátních LV na konkrétní barvy z palety
    unique_lvs = df_parcel_data["LV"].astype(str).unique()
    lv_color_map = {lv: DISTINCT_COLORS[i % len(DISTINCT_COLORS)] for i, lv in enumerate(unique_lvs)}

    items: list[tuple[Polygon, tuple[float, float], str, str]] = []
    legend_list_items = "" # Sem se bude skládat HTML pro plovoucí okno legendy

    # Zpracování geometrie a atributů pro každý řádek (parcelu)
    for idx, row in df_parcel_data.iterrows():
        posList_str = row.get("geometry_posList")

        if not posList_str or pd.isna(posList_str):
            continue

        # Převod textového seznamu souřadnic S-JTSK na GPS (WGS84)
        coords = list(map(float, posList_str.split()))
        xy_pairs = list(zip(coords[0::2], coords[1::2]))
        lon_lat_pairs = [transformer.transform(x, y) for x, y in xy_pairs]

        poly = Polygon(lon_lat_pairs)
        if not poly.is_valid or poly.is_empty:
            continue

        centroid = (poly.centroid.y, poly.centroid.x)

        # Načtení informací pro popisky
        parc_label = row.get("label", "")
        area = row.get("areaValue_m2", None)
        lv = str(row.get("LV", "Neznámé"))
        okres = row.get("okres_nazev", "Neznámý")
        ku = row.get("ku_nazev", "Neznámé")
        
        # Příprava URL odkazu (očištění ID od prefixu "CP.")
        gml_id_raw = str(row.get("gml_id", ""))
        clean_id = gml_id_raw.replace("CP.", "")
        url_kn = f"https://nahlizenidokn.cuzk.gov.cz/ZobrazObjekt.aspx?&typ=parcela&id={clean_id}"

        area_str = f"{float(area):,.0f}".replace(",", " ") if pd.notna(area) else "neznámá výměra"
        color = lv_color_map[lv]

        # Třířádkový HTML popisek nad polygonem mapy
        label_html = f"LV č. {lv}<br>{parc_label}<br>{area_str} m²"
        
        items.append((poly, centroid, label_html, color))

        # Přidání položky do plovoucího seznamu (legendy) s odkazem
        legend_list_items += (
            f"<li style='margin-bottom: 8px; border-bottom: 1px solid #e0e0e0; padding-bottom: 4px;'>"
            f"<span style='display:inline-block; width:14px; height:14px; background-color:{color}; "
            f"border:1px solid #333; margin-right:8px; vertical-align:middle;'></span>"
            f"<span style='vertical-align:middle; font-family:sans-serif;'>"
            f"LV č.{lv}, parc.č. <a href='{url_kn}' target='_blank' style='font-weight:bold; color:#0066cc; text-decoration:none;'>{parc_label}</a>, "
            f"{area_str} m², okres {okres}, k.ú. {ku}"
            f"</span></li>"
        )

    # Inicializace prázdné mapy v případě chyby
    if not items:
        print("Nebyla nalezena žádná validní geometrie, vracím defaultní mapu ČR.")
        return folium.Map(location=[49.8, 15.5], zoom_start=7)

    # Výpočet středu mapy podle načtených parcel
    avg_lat = sum(c[0] for _, c, _, _ in items) / len(items)
    avg_lon = sum(c[1] for _, c, _, _ in items) / len(items)

    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=18, tiles=None, width="100%", height="100%")

    # --- DEFINICE PODKLADOVÝCH VRSTEV ---
    # Minimalistická mapa (ideální pro zvýraznění barevných polygonů, nevyžaduje restrikční hlavičky)
    folium.TileLayer(
        tiles="CartoDB positron",
        name="Základní mapa (světlá)",
        control=True
    ).add_to(m)

    folium.raster_layers.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
        attr="© Google",
        name="Google Maps",
        overlay=False,
        control=True
    ).add_to(m)

    # Oficiální ortofoto ČÚZK
    folium.raster_layers.TileLayer(
        tiles="https://ags.cuzk.gov.cz/arcgis1/rest/services/ORTOFOTO_WM/MapServer/tile/{z}/{y}/{x}",
        name="ČÚZK Ortofoto",
        attr="© ČÚZK",
        overlay=False,
        control=True,
        max_zoom=20,
        min_zoom=6,
        show=False,
    ).add_to(m)

    # Oficiální základní topografická mapa (ZTM ČÚZK) - spolehlivější než mapy.cz / osm
    folium.raster_layers.TileLayer(
        tiles="https://ags.cuzk.gov.cz/arcgis1/rest/services/ZTM_WM/MapServer/tile/{z}/{y}/{x}",
        attr="© ČÚZK",
        name="Základní topografická mapa (ČÚZK)",
        overlay=False,
        control=True,
        show=False,
    ).add_to(m)


    class DynamicArcGISTileLayer(Layer):
        """
        Vlastní třída dědící z folium.map.Layer (zajišťuje správné pořadí renderování v DOMu).
        Využívá anonymní in-line třídu L.TileLayer pro bezpečné vygenerování mapy bez kolize proměnných.
        """
        _template = Template(u"""
            {% macro script(this, kwargs) %}
                var {{ this.get_name() }} = new (L.TileLayer.extend({
                    getTileUrl: function(coords) {
                        var tileSize = 256;
                        // Přepočty pro Web Mercator (EPSG:3857)
                        var initialResolution = 2 * Math.PI * 6378137 / tileSize;
                        var originShift = 2 * Math.PI * 6378137 / 2.0;
                        var resolution = initialResolution / Math.pow(2, coords.z);
                        
                        var minx = coords.x * tileSize * resolution - originShift;
                        var maxx = (coords.x + 1) * tileSize * resolution - originShift;
                        var miny = originShift - (coords.y + 1) * tileSize * resolution;
                        var maxy = originShift - coords.y * tileSize * resolution;
                        
                        var bbox = [minx, miny, maxx, maxy].join(",");
                        
                        // Z url byl odstraněn parametr '&layers=show:0', vyžádáme si kompletní sadu vrstev
                        return "{{ this.url }}?bbox=" + bbox +
                            "&bboxSR=102100&imageSR=102100&size=256,256" +
                            "&format=png32&transparent=true&f=image";
                    }
                }))({
                    opacity: {{ this.opacity }},
                    minZoom: {{ this.min_zoom }},
                    maxZoom: {{ this.max_zoom }}
                });
                
                // Bezpečné přidání do parent FeatureGroup
                {{ this.get_name() }}.addTo({{ this._parent.get_name() }});
            {% endmacro %}
        """)

        def __init__(self, url, opacity=0.5, min_zoom=10, max_zoom=18):
            super().__init__()
            self._name = 'DynamicArcGISTileLayer'
            self.url = url
            self.opacity = opacity
            self.min_zoom = min_zoom
            self.max_zoom = max_zoom

    # Přidání vrstvy do mapy (IPR Praha - export endpoint)
    arcgis_url = "https://gs-pub.praha.eu/arcgis/rest/services/pup/uzemni_plan_platny/MapServer/export"

    dynamic_fg = folium.FeatureGroup(
        name="Územní plán Prahy – plán využití",
        overlay=True,
        control=True,
        show=False,
    )
    dynamic_fg.add_child(DynamicArcGISTileLayer(arcgis_url, opacity=0.5, min_zoom=0, max_zoom=30))
    m.add_child(dynamic_fg)



    # --- DEFINICE ZOBRAZOVANÝCH DAT (POLYGONY A TEXTY) ---
    # Rozdělení do dvou nezávislých vrstev kvůli možnosti vypínat text při oddálení
    fg_poly = folium.FeatureGroup(name="Polygony parcel", show=True)
    fg_text = folium.FeatureGroup(name="Popisky parcel (text)", show=True)

    fg_poly_name = fg_poly.get_name()
    fg_text_name = fg_text.get_name()

    for poly, centroid, label_html, color in items:
        # Vykreslení polygonu s uzávěrem lambda funkce pro zachování správné barvy (c=color)
        gj = folium.GeoJson(
            data=poly.__geo_interface__,
            style_function=lambda feature, c=color: {
                "fillColor": c,
                "color": c,
                "weight": 2,
                "fillOpacity": 0.4,
            },
        ).add_to(fg_poly)

        # Vykreslení textového markeru
        mk = folium.Marker(
            location=centroid,
            draggable=True,
            icon=folium.DivIcon(
                icon_size=(150, 54),
                icon_anchor=(75, 27),
                html=(
                    '<div style="font-size:14px; font-weight:bold; line-height: 1.2;'
                    'text-align:center; color: black; '
                    'text-shadow: 2px 2px 4px white, -1px -1px 0 white, 1px -1px 0 white, -1px 1px 0 white, 1px 1px 0 white;">'
                    f"{label_html}"
                    "</div>"
                ),
            ),
        ).add_to(fg_text)

        # Propojení click eventu - smaže prvek z obou vrstev
        gj.add_child(BindClickRemove(
            fg_poly_name=fg_poly_name, 
            fg_text_name=fg_text_name, 
            gj_name=gj.get_name(), 
            mk_name=mk.get_name()
        ))

    fg_poly.add_to(m)
    fg_text.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)

    # --- INJEKCE PLOVOUCÍ LEGENDY ---
    legend_html_container = f"""
    <div style="position: fixed; 
                bottom: 30px; left: 30px; width: auto; max-width: 1250px; max-height: 1000px; 
                background-color: rgba(255, 255, 255, 0.95); border: 2px solid #aaa; z-index: 9999; 
                overflow-y: auto; padding: 10px; border-radius: 8px; box-shadow: 3px 3px 10px rgba(0,0,0,0.3);">
        <h4 style="margin-top: 0; margin-bottom: 10px; font-family: sans-serif; border-bottom: 2px solid #444; padding-bottom: 5px;">
            Zobrazené pozemky
        </h4>
        <ul style="list-style-type: none; padding-left: 0; margin: 0; font-size: 10px;">
            {legend_list_items}
        </ul>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html_container))

    return m

# =============================================================================
# 2. FUNKCE PRO DOTAZOVÁNÍ RÚIAN A INSPIRE (ZÁKLADNÍ BEZE ZMĚN)
# =============================================================================

def convert_to_gps(x: float, y: float, source_epsg: str = "EPSG:5514") -> tuple[float, float]:
    transformer = Transformer.from_crs(source_epsg, "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(x, y)
    return lon, lat

def get_parcel_data(okres_nazev: str, kat_uzemi_nazev: str, parcel_number: str) -> pd.DataFrame:
    PRAGUE_OBEC_KOD = 554782
    PRAGUE_VUSC_KOD = 19
    PRAGUE_ALIASES = {"praha", "hlavni mesto praha", "hlavní město praha", "praha-mesto", "praha město"}

    def _norm(s: str) -> str: return (s or "").strip().lower()
    def _is_prague_okres(name: str) -> bool: return _norm(name) in PRAGUE_ALIASES
    def _sql_escape(s: str) -> str: return (s or "").replace("'", "''")

    base_url_candidates = [
        "https://ags.cuzk.gov.cz/arcgis/rest/services/RUIAN/Prohlizeci_sluzba_nad_daty_RUIAN/MapServer",
        "https://ags.cuzk.cz/ArcGIS/rest/services/RUIAN/MapServer",
    ]

    session = requests.Session()

    def arcgis_query(base_url: str, layer_id: int, where: str, out_fields: str = "*") -> dict:
        params = {"where": where, "outFields": out_fields, "returnGeometry": "false", "f": "json"}
        r = session.get(f"{base_url}/{layer_id}/query", params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        if isinstance(data, dict) and data.get("error"):
            msg = data["error"].get("message", "ArcGIS error")
            raise RuntimeError(f"ArcGIS query error: {msg}")
        return data

    def arcgis_query_first_ok(layer_id: int, where: str, out_fields: str = "*") -> dict:
        last_err = None
        for base_url in base_url_candidates:
            try:
                return arcgis_query(base_url, layer_id, where, out_fields=out_fields)
            except Exception as e:
                last_err = e
        raise RuntimeError(f"Nepodařilo se dotázat RÚIAN ArcGIS. Poslední chyba: {last_err}")

    praha_mode = _is_prague_okres(okres_nazev)
    okres_kod, okres_attrs = None, None

    if not praha_mode:
        okres_data = arcgis_query_first_ok(layer_id=15, where=f"nazev = '{_sql_escape(okres_nazev)}'", out_fields="*")
        if not okres_data.get("features"):
            raise ValueError(f"Okres '{okres_nazev}' nebyl nalezen.")
        okres_attrs = okres_data["features"][0]["attributes"]
        okres_kod = okres_attrs.get("kod")

    ku_data = arcgis_query_first_ok(layer_id=7, where=f"nazev LIKE '{_sql_escape(kat_uzemi_nazev)}%'", out_fields="kod,nazev,obec")
    if not ku_data.get("features"):
        raise ValueError(f"Katastrální území podobné '{kat_uzemi_nazev}' nebylo nalezeno.")

    valid_ku = []
    for f in ku_data["features"]:
        ku_atr = f["attributes"]
        obec_kod = ku_atr.get("obec")
        if obec_kod is None: continue

        obec_data = arcgis_query_first_ok(layer_id=12, where=f"kod = {int(obec_kod)}", out_fields="kod,nazev,okres")
        if not obec_data.get("features"): continue
        obec_atr = obec_data["features"][0]["attributes"]

        if praha_mode:
            if int(obec_atr.get("kod", -1)) == PRAGUE_OBEC_KOD:
                valid_ku.append({"ku_kod": ku_atr.get("kod"), "ku_nazev": ku_atr.get("nazev"), "obec_kod": obec_atr.get("kod"), "obec_nazev": obec_atr.get("nazev")})
        else:
            if obec_atr.get("okres") == okres_kod:
                valid_ku.append({"ku_kod": ku_atr.get("kod"), "ku_nazev": ku_atr.get("nazev"), "obec_kod": obec_atr.get("kod"), "obec_nazev": obec_atr.get("nazev")})

    if not valid_ku:
        raise ValueError(f"Nebylo nalezeno žádné katastrální území.")

    selected_ku = valid_ku[0]

    if praha_mode:
        vusc_kod = PRAGUE_VUSC_KOD
        vusc_data = arcgis_query_first_ok(layer_id=17, where=f"kod = {int(vusc_kod)}", out_fields="kod,nazev")
        vusc_attrs = vusc_data["features"][0]["attributes"]
        okres_kod_out = None
        okres_nazev_out = okres_nazev
    else:
        okres2 = arcgis_query_first_ok(layer_id=15, where=f"kod = {int(okres_kod)}", out_fields="kod,nazev,vusc")
        okres_attrs = okres2["features"][0]["attributes"]
        vusc_kod = okres_attrs.get("vusc")
        vusc_data = arcgis_query_first_ok(layer_id=17, where=f"kod = {int(vusc_kod)}", out_fields="kod,nazev")
        vusc_attrs = vusc_data["features"][0]["attributes"]
        okres_kod_out = okres_attrs.get("kod")
        okres_nazev_out = okres_attrs.get("nazev")

    # WFS INSPIRE GetParcel
    params_wfs = {
        "service": "WFS", "version": "2.0.0", "request": "GetFeature",
        "storedQuery_id": "GetParcel", "UPPER_ZONING_ID": selected_ku["ku_kod"], "TEXT": parcel_number
    }
    resp_wfs = session.get("https://services.cuzk.cz/wfs/inspire-CP-wfs.asp", params=params_wfs, timeout=30)
    resp_wfs.raise_for_status()

    tree = etree.fromstring(resp_wfs.content)
    ns = {"wfs": "http://www.opengis.net/wfs/2.0", "gml": "http://www.opengis.net/gml/3.2", "CP": "http://inspire.ec.europa.eu/schemas/cp/4.0", "base": "http://inspire.ec.europa.eu/schemas/base/3.3"}

    parcel_elem = tree.find(".//CP:CadastralParcel", namespaces=ns)
    if parcel_elem is None:
        raise ValueError(f"Parcela {parcel_number} nebyla nalezena.")

    def get_text(elem, path):
        sub = elem.find(path, namespaces=ns)
        return sub.text.strip() if sub is not None and sub.text else None

    parcel_data = {
        "gml_id": parcel_elem.get("{http://www.opengis.net/gml/3.2}id"),
        "areaValue_m2": float(get_text(parcel_elem, "CP:areaValue") or 0),
        "beginLifespanVersion": get_text(parcel_elem, "CP:beginLifespanVersion"),
        "endLifespanVersion": get_text(parcel_elem, "CP:endLifespanVersion"),
        "label": get_text(parcel_elem, "CP:label"),
        "nationalCadastralReference": get_text(parcel_elem, "CP:nationalCadastralReference"),
        "inspire_localId": get_text(parcel_elem, "CP:inspireId/base:Identifier/base:localId"),
        "inspire_namespace": get_text(parcel_elem, "CP:inspireId/base:Identifier/base:namespace"),
        "refPoint_x": None, "refPoint_y": None, "refPoint_lon": None, "refPoint_lat": None,
        "geometry_posList": get_text(parcel_elem, "CP:geometry/gml:Polygon/gml:exterior/gml:LinearRing/gml:posList"),
        "ku_kod": selected_ku["ku_kod"], "ku_nazev": selected_ku["ku_nazev"],
        "obec_kod": selected_ku["obec_kod"], "obec_nazev": selected_ku["obec_nazev"],
        "okres_kod": okres_kod_out, "okres_nazev": okres_nazev_out,
        "vusc_kod": vusc_attrs.get("kod"), "vusc_nazev": vusc_attrs.get("nazev"),
    }

    ref_point = get_text(parcel_elem, "CP:referencePoint/gml:Point/gml:pos")
    if ref_point:
        coords = ref_point.split()
        if len(coords) >= 2:
            parcel_data["refPoint_x"] = float(coords[0])
            parcel_data["refPoint_y"] = float(coords[1])
            lon, lat = convert_to_gps(float(coords[0]), float(coords[1]))
            parcel_data["refPoint_lon"], parcel_data["refPoint_lat"] = lon, lat

    return pd.DataFrame([parcel_data])

# =============================================================================
# 3. HLAVNÍ BLOK: ZPRACOVÁNÍ VSTUPŮ A VÝSTUP
# =============================================================================

# Vstupní data (okres, katastrální území, parcelní číslo, číslo LV)
parcely = [


    ("Praha", "Újezd u Průhonic", "213/14", "245"),
    ("Praha", "Újezd u Průhonic", "213/19", "245"),
    ("Praha", "Újezd u Průhonic", "213/21", "245"),
    ("Praha", "Újezd u Průhonic", "214/17", "245"),
    ("Praha", "Újezd u Průhonic", "214/18", "245"),
    ("Praha", "Újezd u Průhonic", "214/21", "245"),    
    ("Praha", "Újezd u Průhonic", "214/37", "245"),
    ("Praha", "Újezd u Průhonic", "214/38", "245"),    
    ("Praha", "Újezd u Průhonic", "214/43", "245"),
    ("Praha", "Újezd u Průhonic", "214/69", "245"),
    ("Praha", "Újezd u Průhonic", "214/207", "245"),    
    ("Praha", "Újezd u Průhonic", "214/502", "245"),


    ("Praha", "Újezd u Průhonic", "213/17", "544"),
    ("Praha", "Újezd u Průhonic", "213/18", "544"),
    ("Praha", "Újezd u Průhonic", "213/24", "544"),
    ("Praha", "Újezd u Průhonic", "214/36", "544"),
    ("Praha", "Újezd u Průhonic", "214/51", "544"),
    ("Praha", "Újezd u Průhonic", "214/59", "544"),
    ("Praha", "Újezd u Průhonic", "214/66", "544"),
    ("Praha", "Újezd u Průhonic", "214/67", "544"),
    ("Praha", "Újezd u Průhonic", "214/80", "544"),
    ("Praha", "Újezd u Průhonic", "214/81", "544"),
    ("Praha", "Újezd u Průhonic", "214/82", "544"),
    ("Praha", "Újezd u Průhonic", "214/83", "544"),
    ("Praha", "Újezd u Průhonic", "214/84", "544"),
    ("Praha", "Újezd u Průhonic", "214/503", "544"),
    ("Praha", "Újezd u Průhonic", "626/12", "544"),
    ("Praha", "Újezd u Průhonic", "674/5", "544"),




    ("Praha", "Šeberov", "537/14", "813"),

    ("Praha", "Šeberov", "537/28", "812"),

    ("Praha", "Křeslice", "413/18", "72"),
    # Přidejte další parcely podle potřeby...
]

dfs = []

print("Načítám data z API ČÚZK...")
# Iterace skrz vstupní tuple o čtyřech prvcích
for okres, ku, parc, lv in parcely:
    df_one = get_parcel_data(okres, ku, parc)
    
    # Přidání LV jako nového sloupce pro další zpracování a export
    df_one["LV"] = str(lv) 
    
    dfs.append(df_one)

# Sloučení všech nalezených parcel do jednoho DataFramu
df_parcel_data = pd.concat(dfs, ignore_index=True)

# Příprava dat pro Excel
df_export = (
    df_parcel_data
    .assign(
        parcelni_cislo=lambda d: d["label"],
        lat=lambda d: d["refPoint_lat"],
        lon=lambda d: d["refPoint_lon"],
    )
    # Zahrnutí sloupce LV do konečného exportu
    .loc[:, ["okres_nazev", "ku_nazev", "obec_nazev", "parcelni_cislo", "LV", "lat", "lon"]]
    .rename(columns={
        "okres_nazev": "okres",
        "ku_nazev": "katastralni_uzemi",
        "obec_nazev": "obec",
    })
)

df_export["lat"] = df_export["lat"].round(8)
df_export["lon"] = df_export["lon"].round(8)

# Uložení DataFramu do Excelu
out_path = "parcely_gps.xlsx"
df_export.to_excel(out_path, index=False, sheet_name="parcely_gps")
print(f"Data uložena do: {out_path}")

# Vykreslení interaktivní mapy
print("Generuji mapu...")
m = plot_parcels_on_map(df_parcel_data)

# Uložení mapy do souboru
html_file = "mapa_parcel.html"
m.save(html_file)
print(f"Interaktivní mapa uložena do: {html_file}")

# Zobrazení mapy přímo ve VS Code (Jupyter)
display(m)

Načítám data z API ČÚZK...
Data uložena do: parcely_gps.xlsx
Generuji mapu...
Interaktivní mapa uložena do: mapa_parcel.html
